# 01 — E0: recompute from cache (narrowed scope)

No API calls. **Scope narrowed by decision on 2026-11-12**: `results/raw/` is not available upstream, so the full E0 design (recompute McNemar, Holm-Bonferroni with m=7/m=20, tie-handling split for W4/W6/W8) cannot run. This notebook only reconciles the aggregated cache against the paper's own published tables (Table 1, Table 2) and states plainly which W-items stay open.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
UPSTREAM = PROJECT_ROOT / "data" / "upstream"

from src.data_audit import load_aggregated_metrics, reconcile_table1

aggregated = load_aggregated_metrics(UPSTREAM)
report = reconcile_table1(aggregated, UPSTREAM / "analysis" / "figures" / "table1_main_results.csv")
report

In [ ]:
# Table 2 (bias scores) is loaded for reference only -- it is NOT independently
# recomputed. Recomputing a bias score needs per-instance verdicts on the
# controlled STYLE/LENGTH/POSITION pairs, which lives in results/raw/ (unavailable).
table2 = pd.read_csv(UPSTREAM / "analysis" / "figures" / "table2_bias_summary.csv")
print("Table 2 as published upstream (not independently verified here):")
table2

In [ ]:
# W8 (partial): the tie-rate half of the claim IS checkable from aggregated
# verdict_distribution counts alone -- no per-instance data needed for this part.
from src.data_audit import tie_rate

s1_tie = tie_rate(aggregated, "claude-sonnet-4", "S1_position_swap", "mt_bench")
s8_tie = tie_rate(aggregated, "claude-sonnet-4", "S8_combined_budget", "mt_bench")
print(f"Claude Sonnet 4 tie rate -- S1: {s1_tie:.1%}, S8: {s8_tie:.1%}")
print("Paper's claim: S1 32.8% tie, S8 12.5% tie -- ", "MATCHES" if abs(s1_tie-0.328)<0.005 and abs(s8_tie-0.125)<0.005 else "DOES NOT MATCH")

## W-item status after this notebook

| ID | Trạng thái sau E0 (thu hẹp) |
| --- | --- |
| W4 (Holm-Bonferroni m=7 vs m=20, p-value mismatch) | **Chưa kết luận được** — cần Table 4/5 hoặc Phụ lục D của bài gốc, hoặc per-instance p-values; không có trong cache aggregated |
| W5 (claim 2 mâu thuẫn nội bộ về dấu bias) | Đã xác minh bằng đọc văn bản trước đó, không cần dữ liệu |
| W6 (khoảng giảm S1 trên LLMBar: văn bản vs Table 2) | **Chưa kết luận được** — cần giá trị gốc theo từng cặp; bảng Table 2 ở đây là bias score, không phải khoảng giảm agreement |
| W8 (mức tăng +11,5 của Claude, tách theo tie) | **Một phần đã xác nhận**: tỷ lệ tie S1→S8 (32,75%→12,5%) khớp chính xác với con số bài nêu, tính thẳng từ verdict_distribution. Phần còn lại (agreement rate loại trừ tie) vẫn **chưa kết luận được** — cần verdict đúng/sai theo từng mẫu cụ thể, không có trong aggregated |

Table 1 reconciliation (this notebook) passed: aggregated cache agrees with the paper's published MT-Bench numbers within 0.005. That, plus the W8 tie-rate check, is what this narrowed E0 scope can actually close out.